# Module 7 - Build the GPT (PyTorch, on Colab)

[![Open In Colab](https://colab.research.google.com/assets/colab-badge.svg)](https://colab.research.google.com/github/Vritti-Dev/foundation-model/blob/main/notebooks/m07_gpt.ipynb)

## What you'll build

You've built every piece in NumPy. Now we assemble the real thing in
**PyTorch**: a small decoder-only GPT, nanoGPT-style. PyTorch gives us
autograd (the Module 3 idea, industrial strength), GPU support, and
battle-tested layers, so we can stack the components into a working
transformer and train it.

> **Run this on Colab.** It uses PyTorch and benefits from a GPU. Click
> the *Open in Colab* badge above. A free **Google account** is
> required. Grading is by **pasting the printed token** into the browser
> grader -- the notebook never uploads your weights.

The reference `GPT` (in `reference/gpt/model.py`) is a stack of pre-LayerNorm
blocks: token + positional embeddings, then `n_layer` blocks of causal
self-attention + MLP with residual connections, a final LayerNorm, and a
linear language-model head. The `CONFIGS['cpu']` preset is a ~0.8M
parameter model that runs fine even without a GPU.

In [ ]:
# Bootstrap (run me first / safe to re-run): clone the course repo on
# Colab and put it on sys.path so `from reference...` and `from grader...`
# work in every cell below. No-op when running locally.
import os, sys
try:
    import google.colab  # noqa: F401
    _repo_dir = '/content/foundation-model'
    if not os.path.exists(_repo_dir):
        !git clone -q https://github.com/Vritti-Dev/foundation-model.git {_repo_dir}
    if _repo_dir not in sys.path:
        sys.path.insert(0, _repo_dir)
    os.chdir(_repo_dir)
    print('course repo ready:', _repo_dir)
except ImportError:
    pass  # local: cwd already on sys.path

# Device detection: use a GPU if Colab gave you one, else fall back to CPU.
import torch
device = 'cuda' if torch.cuda.is_available() else 'cpu'
print('device:', device)


In [ ]:
# Build the reference GPT from the CPU-sized config and sanity-check it.
import torch
from reference.gpt.model import GPT, CONFIGS

cfg = CONFIGS['cpu']
model = GPT(cfg).to(device)
n_params = sum(p.numel() for p in model.parameters())
print(f'parameters: {n_params:,}')

# Forward a batch of zeros to confirm the output shape (B, T, vocab).
x = torch.zeros(2, cfg.block_size, dtype=torch.long, device=device)
logits, loss = model(x)
print('logits shape:', tuple(logits.shape), '| loss (no targets):', loss)


In [ ]:
# A quick training sanity check: a few AdamW steps on a random batch
# should drive the loss down, confirming autograd + the model wiring work.
import torch
torch.manual_seed(0)
x = torch.randint(0, cfg.vocab_size, (4, cfg.block_size), device=device)
y = torch.randint(0, cfg.vocab_size, (4, cfg.block_size), device=device)
_, l0 = model(x, y)
opt = torch.optim.AdamW(model.parameters(), lr=1e-3)
for _ in range(20):
    opt.zero_grad(); _, l = model(x, y); l.backward(); opt.step()
_, l1 = model(x, y)
print(f'loss {l0.item():.4f} -> {l1.item():.4f}')


## Print your submission token

The final cell trains the model briefly and prints a one-line
**submission token**. Copy that line and paste it into the browser
grader for Module 7. The token encodes your final loss (graded by a
tolerance band), an architecture fingerprint, and a deterministic
sample hash -- never your raw weights.

In [ ]:
# Train briefly and print the submission token for Module 7.
from reference.gpt.train import train, make_submission_token

# A small built-in corpus so the notebook is self-contained.
data = (
    'First Citizen:\nBefore we proceed any further, hear me speak.\n'
    'All:\nSpeak, speak.\nFirst Citizen:\nYou are all resolved rather to '
    'die than to famish?\nAll:\nResolved. resolved.\n'
) * 8

res = train(config='cpu', max_iters=50, seed=1337, data=data)
print(make_submission_token(res, 'M7'))


## Going deeper

* Karpathy, **nanoGPT** (the reference codebase): <https://github.com/karpathy/nanoGPT>
* Karpathy, **Let's build GPT from scratch** (Zero to Hero #4): <https://www.youtube.com/watch?v=kCc8FmEb1nY>
* Vaswani et al., **Attention Is All You Need** (the transformer paper, 8 pages): <https://arxiv.org/abs/1706.03762>
* Jay Alammar, **The Illustrated Transformer**: <https://jalammar.github.io/illustrated-transformer/>
* Radford et al., **GPT-1 paper** (very readable): <https://cdn.openai.com/research-covers/language-unsupervised/language_understanding_paper.pdf>
